# Week 7 v2 — Notebook 2B: DTD on every image (shrink512 and full)

**What it does:** reads the dataset `PATH_TO_W7_COMMON_SETS_DATASET` and runs the pre-trained **DTD** (Document Tampering Detector, DocTamper
weights) on **every image** of the manifest at two sizes: `shrink512` (long side 512 px) and `full` (full resolution, in
512 × 512 pieces, exactly as the authors' T-SROIE script does it). It saves one **result map per image and size** in the
standard format that notebook 3 reads. This is a separate notebook from 2A because DTD needs its own Python environment.

**How to run (Kaggle):** attach the datasets `PATH_TO_W7_COMMON_SETS_DATASET` and `PATH_TO_DTD_WEIGHTS_DATASET` (the three files `dtd_doctamper.pth`,
`swin_imagenet.pt`, `vph_imagenet.pt`); Accelerator = **GPU T4 x2**; Internet **ON**. First run with `SMOKE = True`
(3 pages per set), read the printed estimate, then set `SMOKE = False` in the first code cell and use **Save & Run All**.

**What you get:** zip 3 (`w7_2B_3_NEXT_ROUND_dtd-results.zip`) = the folder `dtd/` with `run_info.json`, `run_log.csv` and the
maps. Upload it as a Kaggle dataset called `dtd-results`; notebook 3 finds it by itself.

**Result format (STANDARD):** `results/dtd/<size>/<image set>/<version>/<format>/<page>.png`, an 8-bit greyscale PNG the SAME
size as the page, value = round(255 × probability of "tampered").

**How DTD reads an image (important for the write-up):** DTD needs a **JPEG file**, because it reads the JPEG's DCT
coefficients and quantisation table (with `jpegio`) as well as the pixels.
- A **JPG page at full size** is used as it is: pieces of pixels and coefficients are cropped straight from the file.
- A **PNG page at full size**, and **every shrink512 page** (resizing breaks the JPEG grid), is first saved as a **JPEG of
  quality 100** (4:4:4, the closest thing to no compression) and then read. The notebook counts these files.
- Pieces follow the authors' `infer_sroie.py`: 512 × 512 pieces from (0, 0); at the right and bottom edge the last piece is
  taken from the edge and **overlaps** the previous one (no white padding); pages smaller than 512 are padded first.

## 1. CONFIG

**Why (ELI5):** every setting in one place. `SMOKE=True` runs 3 pages per set so that a mistake costs minutes, not hours.

In [ ]:
# standard tools for files, text, time and shell commands
import os, re, io, sys, json, glob, time, shutil, zipfile, pickle, shlex, subprocess, random, warnings
# the date for the logs
from datetime import datetime
# arrays and tables
import numpy as np
import pandas as pd
# image reading and writing
from PIL import Image
# OpenCV (the same image reader the detectors use)
import cv2

# ======================= CONFIG: change settings here and nowhere else =======================
# which detector this notebook runs (it names the result folder)
DETECTOR = "dtd"
# True = only SMOKE_N pages per set (a quick test); it MUST be False for the real run (Save & Run All)
SMOKE = False
# pages per set in a smoke run
SMOKE_N = 3
# seed for every random choice
SEED = 42
# the sizes to run: shrink512 = long side 512; full = full resolution in 512 x 512 pieces
SIZES = ["shrink512", "full"]
# piece side (the detector's own input size)
PIECE = 512
# images handled per chunk (a chunk is prepared, run, stitched and then deleted, so the disk never fills up)
CHUNK_IMAGES_FULL = 100
# the same for shrink512 (one small image per page, so bigger chunks)
CHUNK_IMAGES_SHRINK = 400
# the notebook stops if its own timed estimate is longer than this many minutes (a smoke run never stops)
MAX_RUN_MINUTES = 150
# how many unique images the real full run has (notebook 1: 970 TextCtrL + 356 shared49 + 98 AnyText + 98 DiffSTE AI edits)
EXPECTED_IMAGES_FULL_RUN = 1522
# optional: one folder that holds ALL inputs (only used for tests; leave empty on Kaggle)
INPUT_ROOT_OVERRIDE = ""
# ===============================================================================================

# are we on Kaggle?
IN_KAGGLE = os.path.exists("/kaggle/input")
# Kaggle
if IN_KAGGLE:
    # where the datasets are mounted
    SEARCH_ROOTS = ["/kaggle/input"]
    # the folder whose content is saved as notebook output
    WORKING = "/kaggle/working"
    # big temporary files go here (NOT saved as output)
    TMP_ROOT = "/tmp"
# anywhere else (a laptop test)
else:
    # look in the current folder
    SEARCH_ROOTS = [os.getcwd()]
    # outputs next to the notebook
    WORKING = os.path.join(os.getcwd(), "w7_2B_working")
    # temporary files inside the working folder
    TMP_ROOT = os.path.join(WORKING, "tmp")
# an explicit input folder wins
if INPUT_ROOT_OVERRIDE:
    # use it
    SEARCH_ROOTS = [INPUT_ROOT_OVERRIDE]
# the notebook name used in zip names
NB_TAG = "2B"
# the folder that is sent on: results/<detector>/ (maps + run_info.json + run_log.csv)
RES_DIR = os.path.join(WORKING, "results", DETECTOR)
# logs, config, timing, figures of this notebook
RUN_DIR = os.path.join(WORKING, "w7_" + NB_TAG + "_run")
# big temporary files (pieces, probability maps)
WORK_DIR = os.path.join(TMP_ROOT, "w7_" + NB_TAG + "_work")
# the folder for our own helper scripts (run with the detector's venv Python)
SCRIPTS_DIR = os.path.join(WORKING, "dtd_scripts")
# make every folder
for d in [RES_DIR, RUN_DIR, WORK_DIR, SCRIPTS_DIR]:
    # create it
    os.makedirs(d, exist_ok=True)
# seeds
random.seed(SEED)
# numpy seed
np.random.seed(SEED)

# how many GPUs does this machine have? (asked from the driver, so no torch is needed in this kernel)
try:
    # one line per GPU
    gpu_lines = subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True).stdout.strip().splitlines()
# no nvidia-smi
except Exception:
    # no GPU lines
    gpu_lines = []
# at least 1 (a laptop test uses a stub in place of the GPU)
N_GPUS = max(1, len(gpu_lines))
# show the GPUs
print("GPUs:", gpu_lines if gpu_lines else "none found (set Accelerator = GPU T4 x2 on Kaggle)")

# the settings as a dictionary (saved as config.json; the notebook 06 cells below also read from it)
CONFIG = {"detector": DETECTOR, "SMOKE": SMOKE, "SMOKE_N": SMOKE_N, "seed": SEED, "sizes": SIZES, "piece": PIECE, "threshold": 0.5,
          "chunk_images_full": CHUNK_IMAGES_FULL, "chunk_images_shrink": CHUNK_IMAGES_SHRINK, "max_run_minutes": MAX_RUN_MINUTES,
          "n_gpus": N_GPUS, "started": datetime.now().isoformat(timespec="seconds")}

# versions and the repo (the cells copied from notebook 06 read these names)
CONFIG.update({"python_version": "3.10", "torch_version": "1.13.1+cu117", "timm": "0.4.12", "segmentation_models_pytorch": "0.2.1",
               "efficientnet_pytorch": "0.7.1", "albumentations": "1.3.0", "repo_url": "https://github.com/qcf-568/DocTamper"})
# where the DTD code is cloned (the scripts live in its models/ folder)
REPO_DIR = os.path.join(WORKING, "DocTamper-code")
# DTD's own folder: dtd.py loads vph_imagenet.pt and swin_imagenet.pt from the current folder, so scripts run here
MODELS_DIR = os.path.join(REPO_DIR, "models")
# the Python 3.10 venv lives in /tmp so it is NOT saved as notebook output
VENV_DIR = "/tmp/venv_dtd"
# the venv's Python
VENV_PY = VENV_DIR + "/bin/python"
# folders for file lists, prepared JPEGs and predictions
LIST_DIR = os.path.join(WORK_DIR, "lists")
# score tables of the run (small csv files, kept)
SCORES_DIR = os.path.join(RUN_DIR, "scores")
# folder for prepared JPEG inputs
PREP_DIR = os.path.join(WORK_DIR, "prep")
# folder for DTD's raw predictions
PRED_DIR = os.path.join(WORK_DIR, "pred")
# make them
for d in [LIST_DIR, SCORES_DIR, PREP_DIR, PRED_DIR]:
    # create
    os.makedirs(d, exist_ok=True)

# save config.json
with open(os.path.join(RUN_DIR, "config.json"), "w") as f:
    # write it
    json.dump(CONFIG, f, indent=2)

# the list of checks (name, value, passed, hard)
CHECKS = []

# record one check and print it
def check(name, value, passed, hard=True):
    # store it
    CHECKS.append(dict(check=name, value=str(value), passed=bool(passed), hard=hard))
    # show it
    print(("PASS " if passed else ("FAIL " if hard else "WARN ")) + name + " -> " + str(value))

# stop with a clear message if a hard check failed
def stop_if_failed():
    # the hard failures
    failed = [c["check"] for c in CHECKS if c["hard"] and not c["passed"]]
    # stop
    if failed:
        raise RuntimeError("STOP: these checks failed: " + "; ".join(failed))

# a loud warning in smoke mode
if SMOKE:
    # banner
    print("#" * 78)
    print("#  SMOKE=True -> only", SMOKE_N, "pages per set. Set SMOKE=False before Save & Run All.")
    print("#" * 78)
# show the settings
print({k: v for k, v in CONFIG.items() if not isinstance(v, dict)})

## 2. Find the inputs (no typing)

**Why (ELI5):** Kaggle puts each attached dataset in its own folder, sometimes nested. We look for a file we know is inside
each one: `manifest.csv` (the page sets) and the three DTD weight files. The manifest becomes a list of unique images with
the page size read from each file header.

In [ ]:
# folders we never walk into (they hold thousands of images or are our own outputs)
SKIP_DIRS = {"images", "masks", "figures", "cache", "results", "shrink512", "full", "default", "eval_outputs", "w7_2A_working", "w7_2B_working"}

# list every folder (up to max_depth levels down) that contains a file called `marker`
def find_folders_with(marker, max_depth=6):
    # the answer
    found = []
    # look under each search root
    for root in SEARCH_ROOTS:
        # skip roots that do not exist
        if not os.path.isdir(root):
            continue
        # walk down from the root (follow links: Kaggle mounts datasets in two styles)
        for folder, subfolders, files in os.walk(root, followlinks=True):
            # how many levels below the root
            depth = folder[len(root):].count(os.sep)
            # the marker is here
            if marker in files:
                found.append(folder)
            # do not enter big folders or go too deep
            subfolders[:] = [] if depth >= max_depth else [d for d in subfolders if d not in SKIP_DIRS]
    # give back the folders
    return found

# the first folder that holds `marker`, or a clear message (name = what to attach on Kaggle)
def need_folder(marker, name):
    # search
    hits = find_folders_with(marker)
    # nothing
    if not hits:
        raise FileNotFoundError("Cannot find " + marker + ". Attach the dataset " + name + " (Add Input). Looked under: " + str(SEARCH_ROOTS))
    # first hit
    return sorted(hits)[0]

# ---------- the dataset PATH_TO_W7_COMMON_SETS_DATASET ----------
# every folder with a manifest.csv; prefer the one that also holds masks/
cands = find_folders_with("manifest.csv")
# nothing found: stop with a clear message
if not cands:
    raise FileNotFoundError("No manifest.csv found. Attach the dataset PATH_TO_W7_COMMON_SETS_DATASET. Looked under: " + str(SEARCH_ROOTS))
# the real dataset has a masks/ folder next to the manifest
cands.sort(key=lambda p: (not os.path.isdir(os.path.join(p, "masks")), p))
# the dataset folder (manifest paths are relative to it)
SETS_DIR = cands[0]
# read the manifest, every column as text
manifest = pd.read_csv(os.path.join(SETS_DIR, "manifest.csv"), dtype=str, keep_default_na=False)
# the three page sets
SETS = ["textctrl", "anytext", "diffste"]
# SMOKE: keep the first SMOKE_N pages of each set
if SMOKE:
    # pages to keep per set
    keep = {s: sorted(manifest[manifest["set"] == s]["page"].unique())[:SMOKE_N] for s in SETS}
    # filter the rows
    manifest = manifest[[p in keep[s] for s, p in zip(manifest["set"], manifest["page"])]].reset_index(drop=True)
# split the path: images/<image set>/<version>/<format>/<page>.<ext>
parts = manifest["path"].str.split("/", expand=True)
# the folder set the image lives in (textctrl, shared49, anytext or diffste)
manifest["img_set"] = parts[1]
# the file stem (= page id)
manifest["stem"] = parts[4].str.rsplit(".", n=1).str[0]
# one row per unique image file (anytext and diffste rows share the shared49 files)
images = manifest.drop_duplicates("path").reset_index(drop=True)
# full path of each image
images["abs_path"] = [os.path.join(SETS_DIR, p) for p in images["path"]]
# full path of each box mask
images["abs_mask"] = [os.path.join(SETS_DIR, p) for p in images["mask_path"]]
# page sizes, read from the file headers only (width, height)
wh = [Image.open(p).size for p in images["abs_path"]]
# width column
images["W"] = [s[0] for s in wh]
# height column
images["H"] = [s[1] for s in wh]
# a running number (used to name temporary files)
images["idx"] = range(len(images))
# print what we found
print("dataset folder :", SETS_DIR)
print("manifest rows  :", len(manifest), "| unique image files:", len(images))
print(images.groupby(["img_set", "version", "format"]).size().unstack(fill_value=0).to_string())
# the pieces a full-size image needs (rows x columns of 512 x 512, padded at the right and bottom)
images["n_rows"] = np.ceil(images["H"] / PIECE).astype(int)
# columns
images["n_cols"] = np.ceil(images["W"] / PIECE).astype(int)
# pieces per image
images["n_pieces"] = images["n_rows"] * images["n_cols"]
# totals
print("images:", len(images), "| full-size pieces in total:", int(images["n_pieces"].sum()), "| mean pieces per image: %.1f" % images["n_pieces"].mean())

# where a result map lives: results/<detector>/<size>/<image set>/<version>/<format>/<page>.png
def result_path(size, row):
    # follow the standard layout of CLAUDE.md
    return os.path.join(RES_DIR, size, row["img_set"], row["version"], row["format"], row["stem"] + ".png")

In [ ]:
# ---------- DTD inputs ----------
# the DocTamper weights (NOT the T-SROIE ones)
DTD_PTH = os.path.join(need_folder("dtd_doctamper.pth", "PATH_TO_DTD_WEIGHTS_DATASET"), "dtd_doctamper.pth")
# the two ImageNet backbones that dtd.py loads with torch.load
SWIN_PT = os.path.join(need_folder("swin_imagenet.pt", "PATH_TO_DTD_WEIGHTS_DATASET"), "swin_imagenet.pt")
# the vision backbone
VPH_PT = os.path.join(need_folder("vph_imagenet.pt", "PATH_TO_DTD_WEIGHTS_DATASET"), "vph_imagenet.pt")
# show
print("dtd_doctamper.pth :", DTD_PTH)
print("swin_imagenet.pt  :", SWIN_PT)
print("vph_imagenet.pt   :", VPH_PT)

## 3. The DTD environment (Python 3.10 venv, packages one at a time)

**Why (ELI5):** DTD needs old libraries (torch 1.13.1, timm 0.4.12, segmentation_models_pytorch 0.2.1, albumentations 1.3.0,
`jpegio`) that would break Kaggle's own Python, so we build a small separate environment in `/tmp` (about 8–10 minutes).
Each package is installed on its own; if the first way fails, the next way is tried, and **every attempt is logged** in
`install_log.csv`. If every way fails, the notebook stops and prints the last lines of the error.

Then we clone the DocTamper code, copy the two backbone files into its `models/` folder (because `dtd.py` loads them from
the current folder), and write two small scripts: one that **checks the setup** (model loads with 0 missing keys, runs on a
512 × 512 piece) and one that **runs DTD the way `infer_sroie.py` does**.

In [ ]:
# [ENV]
# Function: run one shell command, show a short log, stop with the last 30 lines if it fails
def run(cmd, show_tail=3):
    # Start a stopwatch
    t0 = time.time()
    # Run the command in a shell and capture everything it prints
    res = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    # Join normal output and error output, split into lines
    lines = (res.stdout + res.stderr).strip().splitlines()
    # Seconds the command took
    secs = time.time() - t0
    # If it failed: print the last 30 lines and stop the cell
    if res.returncode != 0:
        print("\n".join(lines[-30:]))
        raise RuntimeError(f"FAILED after {secs:.0f}s: {cmd}")
    # If it worked: print the last few lines and the time
    if show_tail:
        print("\n".join(lines[-show_tail:]))
    print(f"OK ({secs:.0f}s): {cmd[:100]}")

# Keep uv's download cache in /tmp (not saved as notebook output)
os.environ["UV_CACHE_DIR"] = "/tmp/uv-cache"
# Copy files instead of linking them (avoids a warning between different disks)
os.environ["UV_LINK_MODE"] = "copy"

# Install uv into Kaggle's Python
run("pip install -q uv", show_tail=0)
# Build the venv only if it is not there yet (resumable)
if not os.path.exists(VENV_PY):
    # uv downloads its own Python 3.10 and adds pip/setuptools/wheel (--seed)
    run(f"uv venv {VENV_DIR} --python {CONFIG['python_version']} --seed")
# Print the venv's Python version
run(f"{VENV_PY} --version", show_tail=1)

In [ ]:
# [ENV]
# the constraints file: versions every later install must respect
CONSTRAINTS = "/tmp/constraints_dtd.txt"
# write it
with open(CONSTRAINTS, "w") as f:
    # torch, torchvision and numpy below 2
    f.write("torch==1.13.1+cu117\ntorchvision==0.14.1+cu117\nnumpy<2\n")
# torch 1.13.1 for CUDA 11.7 (runs on Kaggle's newer driver)
run(f"uv pip install --python {VENV_PY} torch==1.13.1+cu117 torchvision==0.14.1+cu117 --index-url https://download.pytorch.org/whl/cu117")
# ask the venv: torch version, CUDA version, GPU visible?
run(f"{VENV_PY} -c \"import torch; print('torch', torch.__version__, '| CUDA', torch.version.cuda, '| GPU available:', torch.cuda.is_available())\"", show_tail=1)

In [ ]:
# [ENV]
# the install command for the venv (respects the constraints file)
UV = f"uv pip install --python {VENV_PY} -c {CONSTRAINTS}"
# one row per attempt: package, way, status, seconds, last line of the output
INSTALL_LOG = []

# try several ways to install one package; stop with the last lines of the error if all fail
def try_install(name, variants):
    # each way, in order
    for way, cmd in variants:
        # stopwatch
        t0 = time.time()
        # run it
        res = subprocess.run(cmd, shell=True, capture_output=True, text=True)
        # the output lines
        lines = (res.stdout + res.stderr).strip().splitlines()
        # worked?
        ok = res.returncode == 0
        # log the attempt
        INSTALL_LOG.append(dict(package=name, way=way, status="OK" if ok else "FAILED", seconds=round(time.time() - t0, 1), last_line=(lines[-1] if lines else "")[:150]))
        # done
        if ok:
            # say so
            print("OK  ", name, "(" + way + ")", "%.0fs" % (time.time() - t0))
            return
        # failed: show a few lines and try the next way
        print("FAILED", name, "(" + way + ")")
        print("\n".join(lines[-8:]))
    # every way failed
    raise RuntimeError("Could not install " + name + ": every way failed (see the lines above and install_log.csv)")

# build tools for packages that compile (jpegio)
try_install("build tools", [("apt + pip", "apt-get update -qq && apt-get install -y -qq build-essential && " + UV + " 'setuptools<70' wheel cython 'numpy<2'"),
                            ("pip only", UV + " 'setuptools<70' wheel cython 'numpy<2'")])
# the packages, in order, one at a time: (name, [(way, command), ...])
PLAN = [
    ("numpy<2", [("plain", UV + " 'numpy<2'")]),
    ("opencv-python-headless", [("plain", UV + " opencv-python-headless"), ("older opencv", UV + " 'opencv-python-headless<4.12'")]),
    ("timm==0.4.12", [("plain", UV + " timm==0.4.12")]),
    # smp 0.2.1 asks for efficientnet-pytorch 0.6.3, but DTD's requirements want 0.7.1: install smp without its pins, then the rest
    ("segmentation_models_pytorch==0.2.1", [("no deps", UV + " --no-deps segmentation_models_pytorch==0.2.1")]),
    ("pretrainedmodels==0.7.4", [("plain", UV + " pretrainedmodels==0.7.4"), ("no build isolation", UV + " --no-build-isolation pretrainedmodels==0.7.4")]),
    ("efficientnet_pytorch==0.7.1", [("plain", UV + " efficientnet_pytorch==0.7.1"), ("no build isolation", UV + " --no-build-isolation efficientnet_pytorch==0.7.1")]),
    ("albumentations==1.3.0", [("plain", UV + " albumentations==1.3.0")]),
    ("lmdb", [("plain", UV + " lmdb")]),
    ("six", [("plain", UV + " six")]),
    ("tqdm", [("plain", UV + " tqdm")]),
    ("pillow", [("plain", UV + " pillow")]),
    ("jpegio", [("0.2.8", UV + " jpegio==0.2.8"), ("0.2.8 no build isolation", UV + " --no-build-isolation jpegio==0.2.8"),
                ("latest", UV + " jpegio"), ("from GitHub", UV + " --no-build-isolation git+https://github.com/dwgoon/jpegio.git")]),
]
# install them
for name, variants in PLAN:
    # one at a time
    try_install(name, variants)
# the install table
install_df = pd.DataFrame(INSTALL_LOG)
# save it
install_df.to_csv(os.path.join(RUN_DIR, "install_log.csv"), index=False)
# show it
print(install_df[["package", "way", "status", "seconds"]].to_string(index=False))
# numpy must still be below 2
run(f"{VENV_PY} -c \"import numpy; print('numpy', numpy.__version__)\"", show_tail=1)

In [ ]:
# [ENV]
# Clone only the latest version (depth 1), and only if not cloned already (resumable)
if not os.path.isdir(REPO_DIR):
    run(f"git clone -q --depth 1 {CONFIG['repo_url']} {REPO_DIR}", show_tail=0)
# Read the exact commit we got
commit = subprocess.run(["git", "-C", REPO_DIR, "rev-parse", "--short", "HEAD"],
                        capture_output=True, text=True).stdout.strip()
print("Repo commit:", commit)
print("Root files :", sorted(os.listdir(REPO_DIR)))

# Save the commit in CONFIG and re-save config.json
CONFIG["repo_commit"] = commit
with open(f"{RUN_DIR}/config.json", "w") as f:
    json.dump(CONFIG, f, indent=2)

In [ ]:
# [ENV]
# dtd.py loads the two backbones from the current folder: copy them next to dtd.py
for src in [SWIN_PT, VPH_PT]:
    # copy (only if not there yet)
    if not os.path.exists(os.path.join(MODELS_DIR, os.path.basename(src))):
        # copy the file
        shutil.copy(src, MODELS_DIR)
# show what is in the models folder (file names only)
print(sorted(os.listdir(MODELS_DIR)))

**Setup-check script:** loads `dtd_doctamper.pth` into the model (stops unless there are **0 missing keys**), then runs one
random 512 × 512 piece (saved as JPEG q100 and read with `jpegio` like the real run) and checks the output shape.

In [ ]:
%%writefile /kaggle/working/dtd_scripts/dtd_check.py
# dtd_check.py: run with the venv Python from inside DocTamper/models. Usage: python dtd_check.py <path to dtd_doctamper.pth>

# standard tools
import sys, os, platform, tempfile
# arrays
import numpy as np
# the weight file path
weights = sys.argv[1]
# the libraries DTD needs, with their versions
import torch, cv2, timm, albumentations, torchvision, jpegio
import segmentation_models_pytorch as smp
from PIL import Image
print("python", platform.python_version(), "| torch", torch.__version__, "| CUDA available", torch.cuda.is_available())
print("cv2", cv2.__version__, "| timm", timm.__version__, "| smp", smp.__version__, "| albumentations", albumentations.__version__, "| jpegio ok")
# dtd.py is in the current folder
sys.path.insert(0, os.getcwd())
# the model class (this also loads vph_imagenet.pt and swin_imagenet.pt from the current folder)
from dtd import seg_dtd
# the GPU if there is one
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# build the model exactly as infer_sroie.py does
model = seg_dtd("", 2).to(device)
# open the checkpoint
ck = torch.load(weights, map_location="cpu")
# the weights are stored under 'state_dict' (as infer_sroie.py reads them)
sd = ck["state_dict"] if isinstance(ck, dict) and "state_dict" in ck else ck
# show the layout
print("checkpoint top-level keys:", list(ck.keys())[:6] if isinstance(ck, dict) else type(ck), "| tensors:", len(sd))
# training used DataParallel, so the names start with 'module.': remove that prefix
sd = {(k[7:] if k.startswith("module.") else k): v for k, v in sd.items()}
# load them and count what did not match
missing, unexpected = model.load_state_dict(sd, strict=False)
# the report line that the notebook reads
print("missing keys: %d | unexpected keys: %d" % (len(missing), len(unexpected)))
# show a few names if something did not match
if missing:
    print("  first missing:", missing[:5])
if unexpected:
    print("  first unexpected:", unexpected[:5])
# stop unless every weight was found
assert len(missing) == 0, "model has weights that the checkpoint does not contain"
# evaluation mode
model.eval()
# one random 512 x 512 colour piece
arr = (np.random.default_rng(0).random((512, 512, 3)) * 255).astype("uint8")
# save as JPEG quality 100 and read it back the way infer_sroie.py does
with tempfile.NamedTemporaryFile(suffix=".jpg") as tmp:
    # write
    Image.fromarray(arr).save(tmp.name, "JPEG", quality=100)
    # pixels with cv2 (BGR) and DCT numbers with jpegio
    bgr = cv2.imread(tmp.name)
    # coefficients and quantisation table of the luma channel
    jp = jpegio.read(tmp.name)
    # copy them
    dct = jp.coef_arrays[0].copy()
    # table
    qt = jp.quant_tables[0].copy()
# the same preparation as infer_sroie.py
toctsr = torchvision.transforms.Compose([torchvision.transforms.ToTensor(), torchvision.transforms.Normalize(mean=(0.485, 0.455, 0.406), std=(0.229, 0.224, 0.225))])
# pixels as a tensor
data = toctsr(Image.fromarray(cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB))).unsqueeze(0).to(device)
# DCT numbers: absolute value, at most 20
dct_t = torch.abs(torch.LongTensor(dct)).clamp(0, 20).unsqueeze(0).to(device)
# the 8 x 8 quantisation table
qs = torch.LongTensor(qt).reshape(1, 1, 8, 8).to(device)
# predict
with torch.no_grad():
    # two class scores per pixel
    out = model(data, dct_t, qs)
# probability of class 1 ("tampered")
prob = torch.softmax(out, 1)[:, 1]
# report
print("output shape", tuple(out.shape), "| probability range %.3f .. %.3f" % (float(prob.min()), float(prob.max())))
# the shape must be 1 x 2 x 512 x 512
assert tuple(out.shape) == (1, 2, 512, 512), "unexpected output shape"
print("ALL CHECKS PASSED")

In [ ]:
# [PATCH] one-time fixes for the old pickled checkpoint (GELU attribute + Swin classes in __main__)
import os

DTD_PY = "/kaggle/working/DocTamper-code/models/dtd.py"
BAK    = DTD_PY + ".bak"

if not os.path.exists(DTD_PY):
    raise FileNotFoundError(DTD_PY + " not found: run this cell AFTER the DocTamper-code setup cell")

# always start from the pristine file
if os.path.exists(BAK):
    original = open(BAK).read()
else:
    original = open(DTD_PY).read()
    open(BAK, "w").write(original)

header = (
    "# --- dtd-pickle-fixes (auto) ---\n"
    "import torch.nn as _nn\n"
    "_nn.GELU.approximate = 'none'\n"
    "import importlib as _il, __main__ as _main\n"
    "try:\n"
    "    _sw = _il.import_module((__package__ + '.swins') if __package__ else 'swins')\n"
    "except Exception:\n"
    "    _sw = _il.import_module('models.swins')\n"
    "for _n in dir(_sw):\n"
    "    if not _n.startswith('__') and not hasattr(_main, _n):\n"
    "        setattr(_main, _n, getattr(_sw, _n))\n"
    "# --- end dtd-pickle-fixes ---\n"
)
open(DTD_PY, "w").write(header + original)
print("dtd.py patched: GELU default + Swin classes injected into __main__")


In [ ]:
# [ENV]
# run the check script with the venv Python from inside DocTamper/models, and capture its output
res = subprocess.run([VENV_PY, SCRIPTS_DIR + "/dtd_check.py", DTD_PTH], cwd=MODELS_DIR, capture_output=True, text=True)
# normal output plus only the last 30 lines of any errors
report = res.stdout + "\n".join(res.stderr.strip().splitlines()[-30:])
# show it
print(report)
# save it so the versions are on record
with open(os.path.join(RUN_DIR, "env_versions.txt"), "w") as f:
    # write
    f.write(report)
# stop if it failed
if res.returncode != 0:
    # message
    raise RuntimeError("dtd_check.py failed: read the last lines printed above")

**The inference script** follows `models/tsroie/infer_sroie.py` step by step: read the JPEG with `cv2` (pixels) and `jpegio`
(`coef_arrays[0]`, `quant_tables[0]`); pad pages smaller than 512 (the authors' way: grey JPEG with a fixed quantisation table);
cut 512 × 512 pieces of the image AND of the coefficients on the page's own grid with the authors' edge handling; run the
model on each piece; take the softmax probability of class 1; combine the pieces back. Three small differences, all for the
notebook's own needs: it takes a list of files, runs on one GPU per process, and keeps the probability as a decimal until
the last step (the authors cast ×255 to 8-bit by cutting off decimals; we round, so that "flagged ≥ 128" means "probability
≥ 0.5" exactly like for the other detectors).

In [ ]:
%%writefile /kaggle/working/dtd_scripts/dtd_infer.py
# dtd_infer.py: DTD inference that follows DocTamper/models/tsroie/infer_sroie.py, for a list of JPEG files.
# Run with the venv Python from inside DocTamper/models.
# Usage: python dtd_infer.py --weights W --list list.json --csv scores.csv [--limit N]
#   list.json = [[input jpeg, output png], ...]; output = 8-bit map the same size as the (padded) input; existing outputs are skipped.

# standard tools
import argparse, os, sys, json, time, csv, tempfile
# arrays, images, torch and the JPEG-coefficient reader
import cv2
import torch
import jpegio
import torchvision
import numpy as np
from PIL import Image

# read the command-line options
ap = argparse.ArgumentParser()
# the DocTamper checkpoint
ap.add_argument("--weights")
# the JSON list of [input, output] pairs
ap.add_argument("--list")
# the score table (one row per image)
ap.add_argument("--csv")
# at most this many images (0 = all)
ap.add_argument("--limit", type=int, default=0)
# parse
args = ap.parse_args()

# dtd.py is in the current folder (DocTamper/models)
sys.path.insert(0, os.getcwd())
# the model class
from dtd import seg_dtd
# the GPU (a laptop test has none)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# build the model like infer_sroie.py
model = seg_dtd("", 2).to(device)
# open the checkpoint
ck = torch.load(args.weights, map_location="cpu")
# the weights under 'state_dict'
sd = ck["state_dict"] if isinstance(ck, dict) and "state_dict" in ck else ck
# remove the 'module.' prefix left by DataParallel training
sd = {(k[7:] if k.startswith("module.") else k): v for k, v in sd.items()}
# load and count what did not match
missing, unexpected = model.load_state_dict(sd, strict=False)
# the report line the notebook reads
print("model ready | missing keys: %d | unexpected keys: %d" % (len(missing), len(unexpected)), flush=True)
# evaluation mode
model.eval()

# the fixed quantisation table infer_sroie.py uses for pages smaller than 512
new_qtb = np.array([[2, 1, 1, 2, 2, 4, 5, 6], [1, 1, 1, 2, 3, 6, 6, 6], [1, 1, 2, 2, 4, 6, 7, 6], [1, 2, 2, 3, 5, 9, 8, 6], [2, 2, 4, 6, 7, 11, 10, 8],
                    [2, 4, 6, 6, 8, 10, 11, 9], [5, 6, 8, 9, 10, 12, 12, 10], [7, 9, 10, 10, 11, 10, 10, 10]], dtype=np.int32).reshape(64,).tolist()


# cut the page and its coefficients into 512 x 512 pieces (copied from infer_sroie.py, without the mask part)
def crop_img(img, jpg_dct, crop_size=512):
    # height, width
    h, w, c = img.shape
    # full pieces per side
    h_grids = h // crop_size
    w_grids = w // crop_size
    # the pieces
    crop_imgs = []
    crop_jpe_dcts = []
    # the full grid
    for h_idx in range(h_grids):
        for w_idx in range(w_grids):
            x1 = w_idx * crop_size
            x2 = x1 + crop_size
            y1 = h_idx * crop_size
            y2 = y1 + crop_size
            crop_imgs.append(img[y1:y2, x1:x2, :])
            crop_jpe_dcts.append(jpg_dct[y1:y2, x1:x2])
    # right edge: pieces taken from the edge (they overlap the previous column)
    if w % crop_size != 0:
        for h_idx in range(h_grids):
            y1 = h_idx * crop_size
            y2 = y1 + crop_size
            crop_imgs.append(img[y1:y2, w - 512:w, :])
            crop_jpe_dcts.append(jpg_dct[y1:y2, w - 512:w])
    # bottom edge
    if h % crop_size != 0:
        for w_idx in range(w_grids):
            x1 = w_idx * crop_size
            x2 = x1 + crop_size
            crop_imgs.append(img[h - 512:h, x1:x2, :])
            crop_jpe_dcts.append(jpg_dct[h - 512:h, x1:x2])
    # bottom-right corner
    if w % crop_size != 0 and h % crop_size != 0:
        crop_imgs.append(img[h - 512:h, w - 512:w, :])
        crop_jpe_dcts.append(jpg_dct[h - 512:h, w - 512:w])
    # done
    return crop_imgs, crop_jpe_dcts, h_grids, w_grids


# put the pieces back together (copied from infer_sroie.py)
def combine_img(imgs, h_grids, w_grids, img_h, img_w, crop_size=512):
    # counter over the pieces
    i = 0
    # the empty page
    re_img = np.zeros((img_h, img_w))
    # the full grid
    for h_idx in range(h_grids):
        for w_idx in range(w_grids):
            x1 = w_idx * crop_size
            x2 = x1 + crop_size
            y1 = h_idx * crop_size
            y2 = y1 + crop_size
            re_img[y1:y2, x1:x2] = imgs[i]
            i += 1
    # right edge pieces
    if w_grids * crop_size < img_w:
        for h_idx in range(h_grids):
            y1 = h_idx * crop_size
            y2 = y1 + crop_size
            re_img[y1:y2, img_w - 512:img_w] = imgs[i]
            i += 1
    # bottom edge pieces
    if h_grids * crop_size < img_h:
        for w_idx in range(w_grids):
            x1 = w_idx * crop_size
            x2 = x1 + crop_size
            re_img[img_h - 512:img_h, x1:x2] = imgs[i]
            i += 1
    # corner piece
    if w_grids * crop_size < img_w and h_grids * crop_size < img_h:
        re_img[img_h - 512:img_h, img_w - 512:img_w] = imgs[i]
    # done
    return re_img


# pixels to a normalised tensor (as in infer_sroie.py)
toctsr = torchvision.transforms.Compose([torchvision.transforms.ToTensor(), torchvision.transforms.Normalize(mean=(0.485, 0.455, 0.406), std=(0.229, 0.224, 0.225))])

# the pairs to do (outputs that exist are skipped, so the run is resumable)
pairs = [(i, o) for i, o in json.load(open(args.list)) if not os.path.exists(o)]
# apply the limit
if args.limit > 0:
    pairs = pairs[:args.limit]
# say how many
print(len(pairs), "images to do", flush=True)
# write the csv header the first time
if not os.path.exists(args.csv):
    # header
    csv.writer(open(args.csv, "w", newline="")).writerow(["image", "seconds", "h", "w", "n_pieces"])
# one image at a time
for k, (img_path, out_path) in enumerate(pairs, 1):
    # stopwatch
    t0 = time.time()
    # pixels (BGR) and the JPEG's coefficients and quantisation table, as in infer_sroie.py
    imgs_ori = cv2.imread(img_path)
    h, w, c = imgs_ori.shape
    jpg_dct = jpegio.read(img_path)
    dct_ori = jpg_dct.coef_arrays[0].copy()
    use_qtb2 = jpg_dct.quant_tables[0].copy()
    # pages smaller than 512: pad with white, save as a grey JPEG with the fixed table, read it again (infer_sroie.py does this)
    if min(h, w) < 512:
        # the original script takes the size from the mask, which has the page's size
        H, W = h, w
        # missing rows
        dh = (512 - H) if H < 512 else 0
        # missing columns
        dw = (512 - W) if W < 512 else 0
        # pad with white
        imgs_ori = np.pad(imgs_ori, ((0, dh), (0, dw), (0, 0)), "constant", constant_values=255)
        # save as grey JPEG with the fixed table and read the coefficients from it
        with tempfile.NamedTemporaryFile(delete=True) as tmp:
            # grey
            imgs_ori = Image.fromarray(imgs_ori).convert("L")
            # save
            imgs_ori.save(tmp, "JPEG", qtables={0: new_qtb})
            # read the coefficients
            jpg = jpegio.read(tmp.name)
            # luma coefficients
            dct_ori = jpg.coef_arrays[0].copy()
            # back to a 3-channel array
            imgs_ori = np.array(imgs_ori.convert("RGB"))
            # the fixed table
            use_qtb2 = jpg.quant_tables[0].copy()
        # the new size
        h, w, c = imgs_ori.shape
    # only whole 8 x 8 blocks can be used: cut the extra rows and columns
    if h % 8 == 0 and w % 8 == 0:
        imgs_d = imgs_ori
        dct_d = dct_ori
    else:
        imgs_d = imgs_ori[0:(h // 8) * 8, 0:(w // 8) * 8, :].copy()
        dct_d = dct_ori[0:(h // 8) * 8, 0:(w // 8) * 8].copy()
    # the quantisation table as a tensor
    qs = torch.LongTensor(use_qtb2)
    # size after cutting
    img_h, img_w, _ = imgs_d.shape
    # the pieces
    crop_imgs, crop_jpe_dcts, h_grids, w_grids = crop_img(imgs_d, dct_d, crop_size=512)
    # one probability map per piece
    img_list = []
    # each piece
    for idx, crop in enumerate(crop_imgs):
        # pixels: BGR -> RGB -> normalised tensor
        data = toctsr(Image.fromarray(cv2.cvtColor(crop, cv2.COLOR_BGR2RGB))).unsqueeze(0).to(device)
        # coefficients: absolute value, at most 20
        dct = torch.abs(torch.LongTensor(crop_jpe_dcts[idx])).clamp(0, 20).unsqueeze(0).to(device)
        # the quantisation table as 1 x 1 x 8 x 8
        qs_b = qs.reshape(1, 1, 8, 8).to(device)
        # predict
        with torch.no_grad():
            # the two class scores
            pred = model(data, dct, qs_b)
            # probability of class 1 (tampered)
            pred = torch.nn.functional.softmax(pred, 1)[:, 1].cpu()
            # keep the decimal probability (rounded to 8-bit at the very end)
            img_list.append(pred[0].numpy().astype(np.float32))
    # put the pieces back together
    ci = combine_img(img_list, h_grids, w_grids, img_h, img_w, crop_size=512).astype(np.float32)
    # add back the rows and columns that were cut for the 8 x 8 blocks (zeros), as infer_sroie.py does
    ci = cv2.copyMakeBorder(ci, 0, h - img_h, 0, w - img_w, cv2.BORDER_CONSTANT, value=[0, 0, 0])
    # 8-bit: round(255 x probability)
    out8 = np.clip(np.rint(ci * 255.0), 0, 255).astype(np.uint8)
    # write to a temporary name, then rename (no half-written files)
    tmp_out = out_path[:-4] + "_tmp.png"
    # write
    cv2.imwrite(tmp_out, out8)
    # rename
    os.replace(tmp_out, out_path)
    # seconds of this image
    secs = time.time() - t0
    # one line in the score table
    with open(args.csv, "a", newline="") as f:
        csv.writer(f).writerow([os.path.basename(out_path).split(".")[0], round(secs, 3), h, w, len(crop_imgs)])
    # progress: first 3, every 25th, last
    if k <= 3 or k % 25 == 0 or k == len(pairs):
        print("[%d/%d] %s  %.2fs  %d pieces  max %d" % (k, len(pairs), os.path.basename(out_path), secs, len(crop_imgs), int(out8.max())), flush=True)
print("done", flush=True)

## 4. Helpers: preparing the JPEG inputs, running both GPUs, post-processing

**Why (ELI5):** DTD wants a JPEG file for every image it looks at, so for each image we prepare that file, hand lists of files
to two copies of the inference script (one per GPU), and turn each prediction into the final page-size map.

- **full, JPG page:** the page's own JPEG file is used as it is (its coefficients are read straight from the file).
- **full, PNG page:** saved as JPEG quality 100 (4:4:4) first (counted and logged).
- **shrink512:** the page is shrunk so that its long side is 512 (same rule as ADCD-Net's loader), padded with white to
  512 × 512, saved as JPEG quality 100, predicted, the padding is cropped off, and the map is resized back to the page size
  with bilinear interpolation.
- Work is done in **chunks** (prepare, run, finish, delete) so the disk stays small; images with a map already are skipped.

In [ ]:
# the run log: one row per image and size (seconds, max value, status)
RUN_LOG = os.path.join(RES_DIR, "run_log.csv")
# write the header once
if not os.path.exists(RUN_LOG):
    # header line
    with open(RUN_LOG, "w") as f:
        f.write("image,size,set,version,format,page,seconds,max_value,status" + chr(10))

# turn a probability map (0..1) into the 8-bit result: round(255 x probability)
def to8(prob):
    # round, clip to 0..255, store as 8-bit
    return np.clip(np.rint(np.asarray(prob, dtype=np.float32) * 255.0), 0, 255).astype(np.uint8)

# save one finished 8-bit map in the standard layout and add its run-log row
def save_result(size, row, map8, seconds, status="ok"):
    # the file path
    out = result_path(size, row)
    # make its folder
    os.makedirs(os.path.dirname(out), exist_ok=True)
    # the map must have exactly the page size
    assert map8.shape == (row["H"], row["W"]), "map size " + str(map8.shape) + " != page size " + str((row["H"], row["W"])) + " for " + row["path"]
    # write to a temporary name first, then rename (a crash never leaves a half-written map)
    tmp = out[:-4] + "_tmp.png"
    # write the 8-bit PNG
    cv2.imwrite(tmp, map8)
    # rename
    os.replace(tmp, out)
    # one line in the run log
    with open(RUN_LOG, "a") as f:
        f.write(",".join([row["path"], size, row["set"], row["version"], row["format"], row["page"], "%.3f" % seconds, str(int(map8.max())), status]) + chr(10))

# the images that still have no result at this size (this is what makes the run resumable)
def todo_rows(size):
    # the rows
    rows = []
    # check each image
    for r in images.to_dict("records"):
        # no result file yet
        if not os.path.exists(result_path(size, r)):
            # keep it
            rows.append(r)
    # the list
    return rows

# a stopwatch note for the timing table (keep lines made by earlier cells)
TIMING = globals().get("TIMING", [])
# add one line to the timing table
def log_time(step, seconds, note=""):
    # store
    TIMING.append(dict(step=step, seconds=round(seconds, 1), note=note))

# these two lists are filled below (extra things for the zips)
EXTRA_ALL_DIRS = globals().get("EXTRA_ALL_DIRS", [])
# the safe extra files (tables without statement text)
EXTRA_SAFE_FILES = ["dtd_preparation_counts.csv", "install_log.csv"]

In [ ]:
# round up
def ceil_div(a, b):
    # integer ceiling
    return -(-a // b)

# environment that shows only GPU number k to a script
def gpu_env(k):
    # copy the environment and set the visible GPU
    return dict(os.environ, CUDA_VISIBLE_DEVICES=str(k))

# counts for the log: how many files were re-saved as JPEG quality 100, how many pages are smaller than 512
PREP_COUNTS = {"png_pages_resaved_q100_full": 0, "all_pages_resaved_q100_shrink512": 0, "jpg_pages_used_as_is_full": 0, "pages_smaller_than_512_padded_by_dtd": 0}

# save an RGB array as JPEG quality 100 with no colour subsampling (the closest thing to no compression)
def save_jpeg100(rgb, path):
    # write
    Image.fromarray(rgb).save(path, "JPEG", quality=100, subsampling=0)

# the size after ADCD-Net's rule "shrink the long side to 512" (nothing happens if the page is already small)
def shrunk_size(H, W):
    # already small
    if max(H, W) <= 512:
        # keep
        return H, W
    # the scale
    sc = 512 / max(H, W)
    # rounded sizes (the same rounding as the albumentations resize)
    return int(H * sc + 0.5), int(W * sc + 0.5)

# prepare the JPEG input of every image of a chunk; items = (id, input jpeg, output png), meta = what the post-processing needs
def make_dtd_items(size, rows):
    # the lists
    items, meta = [], {}
    # each image
    for r in rows:
        # a unique id without dots
        tid = ("f" if size == "full" else "s") + "%05d" % r["idx"]
        # where DTD writes its prediction
        out_png = os.path.join(PRED_DIR, tid + ".png")
        # full size
        if size == "full":
            # a small page is padded by DTD itself (counted)
            if min(r["H"], r["W"]) < 512:
                # count
                PREP_COUNTS["pages_smaller_than_512_padded_by_dtd"] += 1
            # a JPG page: DTD reads the file itself
            if r["format"] == "jpg":
                # the original file
                in_path = r["abs_path"]
                # count
                PREP_COUNTS["jpg_pages_used_as_is_full"] += 1
            # a PNG page: save as JPEG quality 100 first
            else:
                # read the pixels (BGR) and convert to RGB
                rgb = cv2.cvtColor(cv2.imread(r["abs_path"], cv2.IMREAD_COLOR), cv2.COLOR_BGR2RGB)
                # the prepared file
                in_path = os.path.join(PREP_DIR, tid + ".jpg")
                # save it
                save_jpeg100(rgb, in_path)
                # count
                PREP_COUNTS["png_pages_resaved_q100_full"] += 1
            # nothing else to remember
            meta[r["idx"]] = dict(tid=tid, out=out_png)
        # shrink512
        else:
            # read the decoded pixels (BGR)
            img = cv2.imread(r["abs_path"], cv2.IMREAD_COLOR)
            # the shrunk size
            nh, nw = shrunk_size(r["H"], r["W"])
            # resize if needed (bilinear, like the albumentations resize)
            if (nh, nw) != (r["H"], r["W"]):
                # resize to (width, height)
                img = cv2.resize(img, (nw, nh), interpolation=cv2.INTER_LINEAR)
            # a white 512 x 512 canvas
            canvas = np.full((512, 512, 3), 255, np.uint8)
            # the page in the top-left corner
            canvas[:nh, :nw] = img
            # the prepared JPEG
            in_path = os.path.join(PREP_DIR, tid + ".jpg")
            # save as JPEG quality 100
            save_jpeg100(cv2.cvtColor(canvas, cv2.COLOR_BGR2RGB), in_path)
            # count
            PREP_COUNTS["all_pages_resaved_q100_shrink512"] += 1
            # remember the shrunk size (to crop the padding)
            meta[r["idx"]] = dict(tid=tid, out=out_png, nh=nh, nw=nw)
        # one item
        items.append((tid, in_path, out_png))
    # done
    return items, meta

# the lines of the model-load reports: (tag, gpu, missing, unexpected)
LOAD_REPORTS = []

# run the items on all GPUs (or `gpus` of them); returns {id: seconds}
def run_dtd_chunk(tag, items, gpus=None, limit=0):
    # how many GPUs
    gpus = gpus or N_GPUS
    # the jobs
    jobs = []
    # one job per GPU
    for k in range(gpus):
        # every gpus-th item, starting at k
        part = items[k::gpus]
        # nothing for this GPU
        if not part:
            continue
        # names of the list, csv and log
        name = tag + "_gpu" + str(k)
        # the list of [input, output] pairs
        list_path = os.path.join(LIST_DIR, name + ".json")
        # write it
        with open(list_path, "w") as f:
            # JSON
            json.dump([[i, o] for (_, i, o) in part], f)
        # the score table (an old one would be appended to)
        csv_path = os.path.join(SCORES_DIR, name + ".csv")
        # remove it
        if os.path.exists(csv_path):
            os.remove(csv_path)
        # the log
        log_path = os.path.join(SCORES_DIR, name + ".log")
        # open it
        lf = open(log_path, "w")
        # the command (run from inside DocTamper/models)
        cmd = [VENV_PY, SCRIPTS_DIR + "/dtd_infer.py", "--weights", DTD_PTH, "--list", list_path, "--csv", csv_path, "--limit", str(limit)]
        # start the job in the background
        proc = subprocess.Popen(cmd, cwd=MODELS_DIR, stdout=lf, stderr=subprocess.STDOUT, env=gpu_env(k))
        # remember it
        jobs.append(dict(gpu=k, proc=proc, log=log_path, lf=lf, csv=csv_path))
    # wait for the jobs; a progress line about once a minute
    t0, last = time.time(), time.time()
    # while any job runs
    while any(j["proc"].poll() is None for j in jobs):
        # short sleep
        time.sleep(2)
        # progress
        if time.time() - last > 60:
            # count the predictions so far
            print("   ... %.1f min, %d predictions so far" % ((time.time() - t0) / 60, len(glob.glob(PRED_DIR + "/*.png"))), flush=True)
            # reset
            last = time.time()
    # close the logs and check every job
    seconds = {}
    # each job
    for j in jobs:
        # close the log
        j["lf"].close()
        # read the log
        text = open(j["log"]).read()
        # failed: show the last 30 lines and stop
        if j["proc"].returncode != 0:
            # show
            print("".join(text.splitlines(True)[-30:]))
            # stop
            raise RuntimeError("dtd_infer.py failed on GPU " + str(j["gpu"]) + " (last 30 lines above)")
        # the model-load report
        for m in re.finditer(r"missing keys: (\d+) \| unexpected keys: (\d+)", text):
            # remember
            LOAD_REPORTS.append((tag, j["gpu"], int(m.group(1)), int(m.group(2))))
        # seconds per image from the score table
        if os.path.exists(j["csv"]):
            # read it
            t = pd.read_csv(j["csv"])
            # id -> seconds
            for name, s in zip(t["image"], t["seconds"]):
                # store
                seconds[str(name)] = float(s)
    # give the seconds back
    return seconds

# turn one prediction into the final page-size 8-bit map
def finish(size, r, m):
    # read the prediction DTD wrote
    pred = cv2.imread(m["out"], cv2.IMREAD_GRAYSCALE)
    # it must exist
    assert pred is not None, "no prediction for " + r["path"]
    # full size: crop to the page (DTD pads pages smaller than 512)
    if size == "full":
        # it cannot be smaller than the page
        assert pred.shape[0] >= r["H"] and pred.shape[1] >= r["W"], "prediction smaller than the page: " + str(pred.shape)
        # crop
        return pred[:r["H"], :r["W"]]
    # shrink512: the prediction is 512 x 512
    assert pred.shape == (512, 512), "shrink512 prediction has shape " + str(pred.shape)
    # crop the white padding
    p = pred[:m["nh"], :m["nw"]]
    # back to the page size with bilinear interpolation (width, height)
    if p.shape != (r["H"], r["W"]):
        # resize
        p = cv2.resize(p, (r["W"], r["H"]), interpolation=cv2.INTER_LINEAR)
    # done
    return p

# delete the temporary files of one chunk
def cleanup():
    # prepared JPEGs and predictions
    for d in [PREP_DIR, PRED_DIR]:
        # remove the files and make the folder again
        shutil.rmtree(d, ignore_errors=True)
        # re-create
        os.makedirs(d, exist_ok=True)

## 5. Setup check and a timed estimate (this is also the model-load test)

**Why (ELI5):** before the long run we run DTD on a handful of real images, on one GPU, and time them. This proves the model
loads with **0 missing keys**, that it runs on real data, and how long the whole run will take. If the estimate is above
`MAX_RUN_MINUTES` the notebook stops (a SMOKE run only prints the projection for the full run).

**Estimate** = (pieces × seconds per piece + shrunk images × seconds per image) ÷ GPUs + one model load per chunk + preparing time.

In [ ]:
# images still to do at each size
todo_full = todo_rows("full") if "full" in SIZES else []
# shrink512
todo_shrink = todo_rows("shrink512") if "shrink512" in SIZES else []
# pieces still to run
n_pieces_total = int(sum(r["n_pieces"] for r in todo_full))
# show the amount of work
print("still to do: full %d images (%d pieces) | shrink512 %d images" % (len(todo_full), n_pieces_total, len(todo_shrink)))

# defaults
per_piece = per_shrink = load_s = prep_per_piece = 0.0
# a sample of full-size images
if todo_full:
    # the first 8 images (spread over formats and sets by the manifest order)
    sample = todo_full[:8]
    # prepare them (time it)
    t0 = time.time()
    # items
    items_f, meta_f = make_dtd_items("full", sample)
    # prepare seconds per image
    prep_per_image = (time.time() - t0) / max(1, len(sample))
    # run on one GPU
    t0 = time.time()
    # run
    secs_f = run_dtd_chunk("est_full", items_f, gpus=1)
    # wall time
    wall_f = time.time() - t0
    # seconds per piece = total seconds / total pieces of the sample
    per_piece = sum(secs_f.values()) / max(1, sum(r["n_pieces"] for r in sample))
    # model load = wall time minus the time on images
    load_s = max(0.0, wall_f - sum(secs_f.values()))
    # seconds of preparing per piece
    prep_per_piece = prep_per_image * len(sample) / max(1, sum(r["n_pieces"] for r in sample))
    # clean up
    cleanup()
# a sample of shrunk images
if todo_shrink:
    # the first 10
    sample = todo_shrink[:10]
    # items
    items_s, meta_s = make_dtd_items("shrink512", sample)
    # run
    t0 = time.time()
    # one GPU
    secs_s = run_dtd_chunk("est_shrink", items_s, gpus=1)
    # wall
    wall_s = time.time() - t0
    # seconds per image (first one includes warm-up)
    vals = list(secs_s.values())
    # median
    per_shrink = float(np.median(vals[1:] if len(vals) > 1 else vals))
    # model load
    load_s = max(load_s, wall_s - sum(vals))
    # clean up
    cleanup()
# the counts of the estimate run are not part of the real counts: reset
for k in PREP_COUNTS:
    # zero
    PREP_COUNTS[k] = 0

# the model must have loaded with 0 missing keys on every job so far
bad_load = [x for x in LOAD_REPORTS if x[2] != 0]
# hard check
check("model loads with 0 missing keys", str(LOAD_REPORTS[:4]) if LOAD_REPORTS else "no load report found", bool(LOAD_REPORTS) and not bad_load)
# stop if not
stop_if_failed()

# chunks
n_chunks = ceil_div(len(todo_full), CHUNK_IMAGES_FULL) + ceil_div(len(todo_shrink), CHUNK_IMAGES_SHRINK)
# GPU minutes
gpu_min = (n_pieces_total * per_piece + len(todo_shrink) * per_shrink) / N_GPUS / 60
# model loading minutes
load_min = n_chunks * load_s / 60
# preparing minutes (CPU, not parallel with the GPUs); shrink images need about 0.1 s each
prep_min = (prep_per_piece * n_pieces_total + 0.1 * len(todo_shrink)) / 60
# total
est_min = gpu_min + load_min + prep_min
# show
print("seconds per piece %.3f | per shrunk image %.3f | model load %.0f s | GPUs %d" % (per_piece, per_shrink, load_s, N_GPUS))
print("ESTIMATE for what is left: %.1f min (GPU %.1f + model loads %.1f + preparing %.1f), %d chunks" % (est_min, gpu_min, load_min, prep_min, n_chunks))
# a smoke run: project to the full run
if SMOKE and len(images):
    # scale by the number of images
    scale = EXPECTED_IMAGES_FULL_RUN / len(images)
    # chunks in a full run
    n_chunks_full_run = ceil_div(EXPECTED_IMAGES_FULL_RUN, CHUNK_IMAGES_FULL) + ceil_div(EXPECTED_IMAGES_FULL_RUN, CHUNK_IMAGES_SHRINK)
    # minutes
    proj = (gpu_min + prep_min) * scale + n_chunks_full_run * load_s / 60
    # show
    print("PROJECTION for the full run (about %d images, same mix of page sizes as these %d): about %.0f min (+ about 10 min environment). Approved up to %d min."
          % (EXPECTED_IMAGES_FULL_RUN, len(images), proj, MAX_RUN_MINUTES))
    # remember
    log_time("estimate: full-run projection (seconds)", proj * 60, "from the SMOKE sample: %.0f min" % proj)
# remember the estimate
log_time("estimate for this run (seconds)", est_min * 60, "%.3f s/piece, %.3f s/shrunk image" % (per_piece, per_shrink))
# too long: stop (a smoke run never stops here)
if not SMOKE and est_min > MAX_RUN_MINUTES:
    # message
    raise RuntimeError("STOP: estimate %.0f min is over MAX_RUN_MINUTES = %d. Raise it in section 1 only if you accept the time." % (est_min, MAX_RUN_MINUTES))

## 6. The run (resumable)

**Why (ELI5):** now every image goes through DTD. For each chunk: prepare the JPEG inputs, run both GPUs, turn the
predictions into final page-size maps in the standard folders, delete the temporary files. If the notebook stops, run it
again: images that already have a map are skipped.

In [ ]:
# one size: chunk after chunk
def process_size(size):
    # the images that still have no map
    rows = todo_rows(size)
    # nothing to do
    if not rows:
        # say so
        print(size, ": nothing to do (all maps exist)")
        return
    # images per chunk
    n = CHUNK_IMAGES_FULL if size == "full" else CHUNK_IMAGES_SHRINK
    # the chunks
    chunks = [rows[i:i + n] for i in range(0, len(rows), n)]
    # stopwatch of this size
    t_size = time.time()
    # each chunk
    for ci, chunk in enumerate(chunks, 1):
        # stopwatch
        t0 = time.time()
        # a unique name
        tag = "%s_%03d" % (size, ci)
        # prepare the JPEG inputs
        items, meta = make_dtd_items(size, chunk)
        # run both GPUs
        secs = run_dtd_chunk(tag, items)
        # finish and save every image
        for r in chunk:
            # its info
            m = meta[r["idx"]]
            # the page-size 8-bit map
            m8 = finish(size, r, m)
            # save it and log it
            save_result(size, r, m8, secs.get(m["tid"], 0.0))
        # delete the temporary files
        cleanup()
        # progress
        done = sum(len(c) for c in chunks[:ci])
        # print
        print("%s chunk %d/%d: %d images, %.1f min | %d/%d images done, %.1f min so far" %
              (size, ci, len(chunks), len(chunk), (time.time() - t0) / 60, done, len(rows), (time.time() - t_size) / 60), flush=True)
    # record the time of this size
    log_time("run " + size, time.time() - t_size, str(len(rows)) + " images")

# the whole run
t_run = time.time()
# each size
for size in SIZES:
    # run it
    process_size(size)
# record the time
log_time("run total", time.time() - t_run, str(len(images)) + " images x " + str(len(SIZES)) + " sizes")
# the model-load reports must be clean
check("model loaded with 0 missing keys in every job", str(len(LOAD_REPORTS)) + " reports, " + str(len([x for x in LOAD_REPORTS if x[2] != 0])) + " with missing keys", all(x[2] == 0 for x in LOAD_REPORTS))
# save and show the preparation counts (which files were re-saved as JPEG quality 100)
pd.DataFrame([PREP_COUNTS]).to_csv(os.path.join(RUN_DIR, "dtd_preparation_counts.csv"), index=False)
# show
print(PREP_COUNTS)

## 7. Checks, `run_info.json`, timing and the three zips

**Why (ELI5):** a last look that every image has a map at every size, that every map has the page's exact size and is 8-bit,
and that the run log has one row per image and size. Any gap **stops** the notebook. Then three zips: **(1) all outputs**;
**(2) safe for Claude** (tables only); **(3) next round** (the folder `dtd/` that notebook 3 reads: upload it as the
dataset `dtd-results`).

In [ ]:
# ---------- page-count checks: every image, every size ----------
# one row per size: expected, found, missing, wrong size
count_rows = []
# problems found
bad_size, bad_mode = [], []
# each size
for size in SIZES:
    # missing results
    missing = 0
    # each image
    for r in images.to_dict("records"):
        # the file
        p = result_path(size, r)
        # missing
        if not os.path.exists(p):
            # count
            missing += 1
            continue
        # read the header only
        im = Image.open(p)
        # same size as the page?
        if im.size != (r["W"], r["H"]):
            # remember
            bad_size.append(size + ":" + r["path"])
        # 8-bit greyscale?
        if im.mode != "L":
            # remember
            bad_mode.append(size + ":" + r["path"])
    # the row
    count_rows.append(dict(size=size, expected=len(images), found=len(images) - missing, missing=missing))
# show the counts
print(pd.DataFrame(count_rows).to_string(index=False))
# result files per size, set, version and format
res_counts = []
# each size
for size in SIZES:
    # count the files by their position in the standard layout
    for r in images.to_dict("records"):
        # only files that exist
        if os.path.exists(result_path(size, r)):
            # one row
            res_counts.append(dict(size=size, img_set=r["img_set"], version=r["version"], format=r["format"]))
# the table
print(pd.DataFrame(res_counts).groupby(["size", "img_set", "version", "format"]).size().unstack(["size", "format"], fill_value=0).to_string())
# the hard checks
check("every image has a result at every size", str([c for c in count_rows if c["missing"] > 0]), all(c["missing"] == 0 for c in count_rows))
# size check
check("every result map has the page size", str(len(bad_size)) + " bad " + str(bad_size[:3]), len(bad_size) == 0)
# mode check
check("every result map is 8-bit greyscale", str(len(bad_mode)) + " bad " + str(bad_mode[:3]), len(bad_mode) == 0)

# ---------- the run log: one row per image and size ----------
# read it
log = pd.read_csv(RUN_LOG)
# images that have a result file but no log row (a crash between the two writes): add them
have = set(zip(log["image"], log["size"]))
# the missing rows
extra = []
# each size and image
for size in SIZES:
    # each image
    for r in images.to_dict("records"):
        # a log row exists
        if (r["path"], size) in have:
            continue
        # read the map to get its max value
        m = cv2.imread(result_path(size, r), cv2.IMREAD_GRAYSCALE)
        # a row
        extra.append(dict(image=r["path"], size=size, set=r["set"], version=r["version"], format=r["format"], page=r["page"], seconds=np.nan, max_value=int(m.max()), status="existing"))
# add them
if extra:
    # extend the log and save it
    log = pd.concat([log, pd.DataFrame(extra)], ignore_index=True)
    # write
    log.to_csv(RUN_LOG, index=False)
# keep only the last row of every (image, size)
log = log.drop_duplicates(["image", "size"], keep="last")
# save the clean log
log.to_csv(RUN_LOG, index=False)
# one row per image and size is required
check("run_log.csv has one row per image and size", str(len(log)) + " rows, expected " + str(len(images) * len(SIZES)), len(log) == len(images) * len(SIZES))
# all rows ok
check("no failed images in the run log", str(int((log["status"] != "ok").sum() - (log["status"] == "existing").sum())) + " not ok", int(((log["status"] != "ok") & (log["status"] != "existing")).sum()) == 0)
# seconds per image and size
print(log.groupby("size")["seconds"].agg(["count", "mean", "median", "max"]).round(3).to_string())

# ---------- a sanity look: mean 8-bit value inside the box versus outside (information, not a check) ----------
# a seeded sample of at most 150 edited images
rng = np.random.default_rng(SEED)
# the edited ones
edited = images[images["version"] != "untouched"]
# sample
sample = edited.iloc[rng.permutation(len(edited))[:150]]
# rows
san = []
# each sampled image and size
for r in sample.to_dict("records"):
    # box mask
    bm = cv2.imread(r["abs_mask"], cv2.IMREAD_GRAYSCALE) > 127
    # each size
    for size in SIZES:
        # the result map
        m = cv2.imread(result_path(size, r), cv2.IMREAD_GRAYSCALE)
        # one row
        san.append(dict(size=size, version=r["version"], format=r["format"], in_box=float(m[bm].mean()), outside=float(m[~bm].mean()), flagged_in_box=bool((m[bm] >= 128).any())))
# the summary table
sanity = pd.DataFrame(san).groupby(["size", "version", "format"]).agg(n=("in_box", "count"), mean_in_box=("in_box", "mean"), mean_outside=("outside", "mean"), flagged_in_box=("flagged_in_box", "mean")).round(3).reset_index()
# show it
print(sanity.to_string(index=False))
# save it
sanity.to_csv(os.path.join(RUN_DIR, "sanity_box_vs_outside.csv"), index=False)

# ---------- run_info.json (sits directly in results/<detector>/) ----------
# versions line from the environment check, if there is one
env_txt = open(os.path.join(RUN_DIR, "env_versions.txt")).read().strip() if os.path.exists(os.path.join(RUN_DIR, "env_versions.txt")) else ""
# a fingerprint of the manifest this run was made from (notebook 3 and Nazhan's run compare it, so a different dataset version cannot slip in)
import hashlib
# md5 of the whole manifest.csv file (also in a SMOKE run, which only reads part of it)
MANIFEST_MD5 = hashlib.md5(open(os.path.join(SETS_DIR, "manifest.csv"), "rb").read()).hexdigest()
# number of rows in the whole file
MANIFEST_ROWS_TOTAL = sum(1 for _ in open(os.path.join(SETS_DIR, "manifest.csv"))) - 1
# the information
run_info = {"detector": DETECTOR, "manifest_md5": MANIFEST_MD5, "manifest_rows_total": MANIFEST_ROWS_TOTAL, "weights_files": [os.path.basename(DTD_PTH), os.path.basename(SWIN_PT), os.path.basename(VPH_PT)], "piece_size": PIECE, "sizes": SIZES, "n_images": int(len(images)), "n_results": int(len(log)),
            "threshold_for_flagging": 0.5, "value_scale": "round(255 x probability of tampered)", "seed": SEED, "SMOKE": SMOKE,
            "python_and_package_versions": env_txt.splitlines()[:6], "notes": ["follows models/tsroie/infer_sroie.py: cv2 pixels + jpegio coef_arrays[0] and quant_tables[0], 512 x 512 pieces on the page grid with overlapping edge pieces, softmax class 1; probability rounded to 8-bit (the authors floor it); full-size JPG pages used as they are; full-size PNG pages and every shrink512 page saved as JPEG quality 100 (4:4:4) first: " + str(PREP_COUNTS), "shrink512 = long side 512, padded white to 512 x 512, padding cropped, map resized back with bilinear interpolation"], "date": datetime.now().isoformat(timespec="seconds")}
# write it
with open(os.path.join(RES_DIR, "run_info.json"), "w") as f:
    # save
    json.dump(run_info, f, indent=2)
# show it
print(json.dumps(run_info, indent=1)[:1500])

# ---------- checks table and timing table ----------
# save the checks
pd.DataFrame(CHECKS).to_csv(os.path.join(RUN_DIR, "checks.csv"), index=False)
# save the timing
pd.DataFrame(TIMING).to_csv(os.path.join(RUN_DIR, "timing.csv"), index=False)
# show both
print(pd.DataFrame(TIMING).to_string(index=False))
# stop if any hard check failed
stop_if_failed()

# ---------- the three zips ----------
# make a zip from (file, name inside zip) pairs
def make_zip(zip_path, items):
    # write the zip
    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
        # each file
        for src, arc in items:
            # add it
            z.write(src, arc)
    # size in MB
    return os.path.getsize(zip_path) / 1e6

# every file under a folder as (path, name in zip)
def walk_items(folder, prefix):
    # the list
    items = []
    # walk
    for root, dirs, fs in os.walk(folder):
        # each file
        for f in fs:
            # skip leftovers
            if f.endswith("_tmp.png") or f == ".DS_Store":
                continue
            # full path
            full = os.path.join(root, f)
            # name inside
            items.append((full, os.path.join(prefix, os.path.relpath(full, folder))))
    # done
    return items

# (3) NEXT ROUND: the result folder results/<detector>/ (what notebook 3 reads), top folder = the detector name
next_items = walk_items(RES_DIR, DETECTOR)
# (1) ALL outputs: the result folder plus every log, config, table and figure of this notebook (and the OCR masks if there are any)
all_items = walk_items(RES_DIR, "results/" + DETECTOR) + walk_items(RUN_DIR, "w7_" + NB_TAG + "_run")
# extra folders that exist
for extra_dir, name in EXTRA_ALL_DIRS:
    # only if it exists
    if os.path.isdir(extra_dir):
        # add its files
        all_items += walk_items(extra_dir, name)
# (2) SAFE for Claude: tables and text only (no maps, no masks, no images); the run log has page ids but no text from the statements
safe_files = [(RUN_LOG, "w7_safe/run_log.csv"), (os.path.join(RES_DIR, "run_info.json"), "w7_safe/run_info.json")]
# the tables and logs of the run folder
for f in ["checks.csv", "timing.csv", "config.json", "env_versions.txt", "sanity_box_vs_outside.csv"] + EXTRA_SAFE_FILES:
    # only if it exists
    if os.path.exists(os.path.join(RUN_DIR, f)):
        # add
        safe_files.append((os.path.join(RUN_DIR, f), "w7_safe/" + f))
# the zip paths (names tell what is inside)
ZIPS = {"1_ALL_outputs": os.path.join(WORKING, "w7_" + NB_TAG + "_1_ALL_outputs.zip"),
        "2_SAFE_for_claude": os.path.join(WORKING, "w7_" + NB_TAG + "_2_SAFE_for_claude.zip"),
        "3_NEXT_ROUND": os.path.join(WORKING, "w7_" + NB_TAG + "_3_NEXT_ROUND_" + DETECTOR + "-results.zip")}
# build them
for key, items in [("1_ALL_outputs", all_items), ("2_SAFE_for_claude", safe_files), ("3_NEXT_ROUND", next_items)]:
    # make and report
    mb = make_zip(ZIPS[key], items)
    # print
    print("%-20s %8.1f MB %6d files  %s" % (key, mb, len(items), ZIPS[key]))
# download links (relative to the working folder on Kaggle)
try:
    # IPython
    from IPython.display import FileLink, display
    # relative links need the working folder as the current folder
    os.chdir(WORKING)
    # each zip
    for key, path in ZIPS.items():
        # label
        print(key)
        # link
        display(FileLink(os.path.relpath(path, WORKING)))
# not in a notebook
except Exception as e:
    # print the paths
    print("no download links here:", e)
# the reminder
print("Upload zip 3 as a Kaggle dataset named " + DETECTOR + "-results (it unzips to a folder called " + DETECTOR + "/ with run_info.json inside).")
# smoke reminder
if SMOKE:
    # banner
    print("#" * 78)
    print("#  SMOKE=True: this was only a test. Set SMOKE=False before Save & Run All.")
    print("#" * 78)
print("DONE.", DETECTOR, "results:", RES_DIR)